In [ ]:
import fiona
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx
import numpy as np
import pandas as pd
import shapely.geometry
import tqdm
from peilbeheerst_model.peilbeheerst_model.parse_crossings import ParseCrossings

In [ ]:
waterschap_data = {
    "HHNK": "../../Data_postprocessed/Waterschappen/HHNK/Noorderkwartier.gpkg",
}

filename = "test9.gpkg"

In [ ]:
cross = ParseCrossings(waterschap_data["HHNK"])
cross.df_gpkg.keys()

In [ ]:
df_hydro, df_dsf, df_hydro_dsf = cross.find_crossings_with_peilgebieden(
    "hydroobject", group_stacked=True, filterlayer="duikersifonhevel"
)

# standaard bestanden
for layer, df in cross.df_gpkg.items():
    df.to_file(filename, layer=layer)

# Crossings wegschrijven
df_dsf.to_file(filename, layer="crossings_duikersifonhevel")
df_hydro.to_file(filename, layer="crossings_hydroobject")
df_hydro_dsf.to_file(filename, layer="crossings_hydroobject_filtered")

In [ ]:
df_dsf = gpd.read_file(filename, layer="crossings_duikersifonhevel")
df_hydro = gpd.read_file(filename, layer="crossings_hydroobject")
df_hydro_dsf = gpd.read_file(filename, layer="crossings_hydroobject_filtered")

for layername, df in zip(
    ["crossings_hydroobject_wlcorr", "crossings_hydroobject_filtered_wlcorr"],
    [df_hydro, df_hydro_dsf],
):
    # Werk op een kopie
    dfs = df.copy()

    # Voeg waterstanden en objecten toe.
    dfs = cross.add_waterlevels_to_crossings(dfs)
    dfs = cross.find_structures_at_crossings(dfs, "stuw")
    dfs = cross.find_structures_at_crossings(dfs, "gemaal")
    dfs = cross.correct_water_flow(dfs)
    dfs = cross.aggregate_identical_links(dfs)

    # Schrijf weg
    dfs.to_file(filename, layer=layername)

In [ ]:
dfs = gpd.read_file(filename, layer="crossings_hydroobject_filtered_wlcorr")

display(dfs[dfs.agg1_used].shape)
display(dfs[dfs.agg1_used].groupby("flip").count())

In [ ]:
dfs = gpd.read_file(filename, layer="crossings_hydroobject_wlcorr")

display(dfs[dfs.agg1_used].shape)
display(dfs[dfs.agg1_used].groupby("flip").count())

In [ ]:
dfs = gpd.read_file(filename, layer="crossings_hydroobject_wlcorr")

G = networkx.DiGraph()
nodes = np.unique(
    np.hstack(
        [
            dfs[dfs.agg1_used].peilgebied_from.dropna().values,
            dfs[dfs.agg1_used].peilgebied_to.dropna().values,
        ]
    )
)
edges = dfs[dfs.agg1_used][["peilgebied_from", "peilgebied_to"]].copy()
for i, row in enumerate(edges.itertuples()):
    if pd.isnull(row.peilgebied_from):
        edges.at[row.Index, "peilgebied_from"] = f"None_{i}"
    if pd.isnull(row.peilgebied_to):
        edges.at[row.Index, "peilgebied_to"] = f"None_{i}"
edges = edges.values.tolist()

G.add_nodes_from(nodes)
G.add_edges_from(edges)

print(G.number_of_edges())

max(networkx.strongly_connected_components(G), key=len)
test = [
    c for c in sorted(networkx.strongly_connected_components(G), key=len, reverse=True)
]
subg = G.subgraph(test[0]).copy()

peilgebieden = cross.df_gpkg["peilgebied"].set_index("globalid")
pos = {
    n: [
        peilgebieden.at[n, "geometry"].centroid.x,
        peilgebieden.at[n, "geometry"].centroid.y,
    ]
    for n in subg.nodes
}

plt.close("all")
fig, ax = plt.subplots(figsize=(15, 15))
networkx.draw_networkx(
    subg, pos, ax=ax, node_color="lime", edgecolors="darkgreen", font_size=6
)